# Phase 9 - Chapter IV Reporting

This notebook regenerates full-data tables, figures, and the results-and-discussion draft. Quick-run metrics are rejected by the reporting module.

In [ ]:
# Cell 1 - Locate the project root and import notebook dependencies.
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / 'configs' / 'config.yaml'
report_dir = project_root / 'reports' / 'phase9'
print(f'Project root: {project_root}')

In [ ]:
# Cell 2 - Generate the canonical Phase 9 report package from full-data artifacts.
from src.reporting import generate_phase9_report

report_result = generate_phase9_report(
    config_path=config_path,
    output_dir=report_dir,
)

print(f"Draft: {report_result['draft_path']}")
print(f"Manifest: {report_result['manifest_path']}")

In [ ]:
# Cell 3 - Review the full-data aggregate scenario table used in Chapter IV.
scenario_columns = [
    'scenario_label',
    'accuracy',
    'macro_precision',
    'macro_recall',
    'macro_f1',
    'training_seconds',
]
report_result['summary'][scenario_columns]

In [ ]:
# Cell 4 - Review the evidence-based H1-H3 conclusions.
report_result['hypothesis_table']

In [ ]:
# Cell 5 - Preview every figure generated for the Chapter IV draft.
figure_titles = {
    'autoencoder_loss': 'Figure 4.1 - Autoencoder Training and Validation Loss',
    'scenario_metrics': 'Figure 4.2 - Aggregate Scenario Metrics',
    'minority_f1': 'Figure 4.3 - Minority-Class F1-Scores',
    's2_confusion': 'Figure 4.4 - S2 Confusion Matrix',
    'representation_baseline': 'Figure 4.5 - Latent vs Original-Feature Baseline',
}

for figure_key, figure_title in figure_titles.items():
    figure_path = Path(report_result['figure_paths'][figure_key])
    assert figure_path.exists(), f'Missing figure: {figure_path}'
    display(Markdown(f'### {figure_title}'))
    display(Image(filename=str(figure_path)))

In [ ]:
# Cell 6 - Validate full-data provenance and all generated deliverables.
with open(report_result['manifest_path'], 'r', encoding='utf-8') as file:
    manifest = json.load(file)

assert manifest['full_data_only'] is True
assert manifest['quick_run_artifacts_excluded'] is True
assert manifest['test_artifacts_unchanged'] is True
assert manifest['optional_baseline_included'] is True
assert manifest['hypothesis_assessments'] == {
    'H1': 'Supported methodologically',
    'H2': 'Partially supported',
    'H3': 'Supported',
}
assert all(Path(path).exists() for path in report_result['table_paths'].values())
assert all(Path(path).exists() for path in report_result['figure_paths'].values())

print('Phase 9 provenance and deliverable checks passed.')

In [ ]:
# Cell 7 - Render the complete Chapter IV draft with local figures resolved correctly.
def display_markdown_with_local_images(markdown_text, base_dir):
    pending_lines = []

    def display_pending_markdown():
        if pending_lines:
            display(Markdown('\n'.join(pending_lines)))
            pending_lines.clear()

    for line in markdown_text.splitlines():
        if line.startswith('![') and '](' in line and line.endswith(')'):
            display_pending_markdown()
            alt_text, relative_path = line[2:-1].split('](', maxsplit=1)
            image_path = (base_dir / relative_path).resolve()
            if not image_path.exists():
                raise FileNotFoundError(f'Missing report image: {image_path}')
            display(Markdown(f'**{alt_text}**'))
            display(Image(filename=str(image_path)))
        else:
            pending_lines.append(line)

    display_pending_markdown()


draft_path = Path(report_result['draft_path'])
draft_text = draft_path.read_text(encoding='utf-8')
assert 'Quick-run artifacts are explicitly rejected' in draft_text
assert 'H2 is therefore partially supported' in draft_text
display_markdown_with_local_images(draft_text, draft_path.parent)